# 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.

- 채점 기준:
    - 출력 결과 일치: 제출한 코드가 오류 없이 제시된 출력 결과와 일치하거나 맥락상 동일해야 정답으로 인정됩니다.
    - 코드의 다양성 인정: 출력 결과가 맥락상 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# 9주차 과제

In [1]:
!pip install -q langchain langchain-core langchain-openai langchain-google-genai langgraph tavily-python python-dotenv

In [2]:
# API 키 설정

# 로컬 환경: 프로젝트 루트의 .env 파일에 키를 저장하고 아래 코드를 사용
from dotenv import load_dotenv
load_dotenv()

# Colab 환경: 위 로컬 환경 설정을 주석 처리하고 아래 코드를 활성화
# import os
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

True

---
## 문제 1. LangChain 기본 모델 호출

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

# TODO 1: 모델 초기화
model = init_chat_model(model="openai:gpt-4o-mini")

# TODO 2: HumanMessage 생성 및 invoke
msg = HumanMessage(content="LangChain이 무엇인지 한 문장으로 설명해줘.")
response = model.invoke([msg])

# TODO 3: 응답 내용 출력
print(response.content)

LangChain은 다양한 자연어 처리(NLP) 모델과 데이터 소스와의 통합을 용이하게 하여, 복잡한 언어 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.


---
## 문제 2. SystemMessage + HumanMessage 대화 구성

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

# TODO 1: 메시지 리스트 구성
sys_msg = SystemMessage(content="너는 AI 에이전트 전문가야. 모든 답변을 2문장 이내로 해줘.")
human_msg = HumanMessage(content="LangGraph를 설명해줘.")
messages = [sys_msg, human_msg]

# TODO 2: 모델 호출
response = model.invoke(messages)

print(response.content)

LangGraph는 자연어 처리와 머신 러닝 기술을 활용하여 언어 간의 관계를 시각적으로 표현하는 도구입니다. 이 플랫폼은 데이터 간의 연결성과 패턴을 분석하여 사용자에게 인사이트를 제공합니다.


---
## 문제 3. Structured Output

In [5]:
from pydantic import BaseModel, Field

# TODO 1: Pydantic 스키마 정의
class FrameworkInfo(BaseModel):
    name: str = Field(..., description="AI 프레임워크 이름")
    description: str = Field(..., description="한 줄 설명")

# TODO 2: structured output 모델 구성
structured_model = model.with_structured_output(FrameworkInfo)

# TODO 3: 호출
result = structured_model.invoke("LangGraph를 소개해줘.")

print(f"name: {result.name}")
print(f"description: {result.description}")

name: LangGraph
description: LangGraph는 자연어 처리(NLP)와 그래프 기반 모델링을 결합하여 텍스트 데이터를 효과적으로 이해하고 분석하는 데 초점을 맞춘 프레임워크입니다. 이 프레임워크는 언어의 구조적 특성을 활용하여 다양한 NLP 작업, 예를 들어 기계 번역, 감성 분석 및 질문 답변 시스템에서 성능을 향상시키는 것을 목표로 합니다. LangGraph는 그래프 데이터 구조를 사용하여 언어의 복잡한 패턴과 관계를 모델링할 수 있게 하여, 더 나은 정보 검색과 텍스트 요약 등의 응용 분야에서도 경쟁력을 갖추고 있습니다.


---
## 문제 4. LangGraph State 정의

In [6]:
import operator
from typing_extensions import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# TODO: AgentState 정의
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    topic: str
    summary: str

# 검증
assert "messages" in AgentState.__annotations__
assert "topic" in AgentState.__annotations__
assert "summary" in AgentState.__annotations__
print("State 정의 완료:", list(AgentState.__annotations__.keys()))

State 정의 완료: ['messages', 'topic', 'summary']


---
## 문제 5. StateGraph 기본 구성

In [7]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END

# TODO 1: respond 노드 함수
def respond(state: AgentState):
    last_content = state["messages"][-1].content
    reply = last_content + " 에 대해 답변드릴게요."
    return {"messages": [AIMessage(content=reply)]}

# TODO 2: StateGraph 구성
builder = StateGraph(AgentState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

# TODO 3: 컴파일 및 실행
graph = builder.compile()
result = graph.invoke({"messages": [HumanMessage(content="LangGraph")]})
print(result["messages"][-1].content)

LangGraph 에 대해 답변드릴게요.


---
## 문제 6. 조건부 라우팅

In [8]:
from typing_extensions import Literal

# TODO 1: route 함수
def route(state: AgentState) -> Literal["long", "short"]:
    content = state["messages"][-1].content
    return "long" if len(content) >= 10 else "short"

# TODO 2: 노드 함수
def long_response(state: AgentState):
    return {"messages": [AIMessage(content="긴 질문이에요.")]}

def short_response(state: AgentState):
    return {"messages": [AIMessage(content="짧은 질문이에요.")]}

# TODO 3: 그래프 구성 (add_conditional_edges 사용)
builder = StateGraph(AgentState)
builder.add_node("long", long_response)
builder.add_node("short", short_response)
path_map = {"long": "long", "short": "short"}
builder.add_conditional_edges(START, route, path_map)
builder.add_edge("long", END)
builder.add_edge("short", END)

graph = builder.compile()

# TODO 4: 실행
result_long = graph.invoke({"messages": [HumanMessage(content="LangGraph에 대해 자세히 설명해줘")]})
result_short = graph.invoke({"messages": [HumanMessage(content="안녕")]})
print(result_long["messages"][-1].content)
print(result_short["messages"][-1].content)

긴 질문이에요.
짧은 질문이에요.


---
## 문제 7. Tool 정의 및 tool_node 구현

In [9]:
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import ToolMessage, AIMessage
from typing_extensions import Annotated

# TODO 1: word_count_tool 정의
@tool(parse_docstring=True)
def word_count_tool(
    text: str,
) -> str:
    """텍스트의 단어 수를 반환한다.

    Args:
        text: 단어 수를 셀 텍스트
    """
    words = text.split()
    count = len(words)
    max_words = 100
    return str(count if count < max_words else max_words)

# TODO 2: tools_by_name 구성
tools = [word_count_tool]
tools_by_name = {tool.name: tool for tool in tools}

# TODO 3: tool_node 구현
def tool_node(state):
    last_msg = state["messages"][-1]
    tool_calls = last_msg.tool_calls
    outputs = []
    for tc in tool_calls:
        tool_fn = tools_by_name[tc["name"]]
        output = tool_fn.invoke(tc["args"])
        outputs.append(
            ToolMessage(content=str(output), name=tc["name"], tool_call_id=tc["id"])
        )
    return {"messages": outputs}

# 검증
fake_ai_msg = AIMessage(
    content="",
    tool_calls=[{"name": "word_count_tool", "args": {"text": "Hello LangGraph"}, "id": "tc1", "type": "tool_call"}]
)
test_result = tool_node({"messages": [fake_ai_msg]})
assert len(test_result["messages"]) == 1
assert isinstance(test_result["messages"][0], ToolMessage)
print("tool_node 테스트 통과")
print("단어 수 결과:", test_result["messages"][0].content)

tool_node 테스트 통과
단어 수 결과: 2


---
## 문제 8. output_schema 적용 및 실행 검증

In [10]:
# TODO 1: ResearchState 정의
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    result: str

# TODO 2: ResearchOutputState 정의
class ResearchOutputState(TypedDict):
    result: str

# TODO 3: llm_call 노드
def llm_call(state: ResearchState):
    msgs = state["messages"]
    response = model.invoke(msgs)
    return {"messages": [response]}

# TODO 4: finalize 노드
def finalize(state: ResearchState):
    last_msg = state["messages"][-1]
    return {"result": last_msg.content}

# TODO 5: StateGraph 구성 (output_schema 적용)
research_builder = StateGraph(ResearchState, output_schema=ResearchOutputState)
research_builder.add_node("llm_call", llm_call)
research_builder.add_node("finalize", finalize)
research_builder.add_edge(START, "llm_call")
research_builder.add_edge("llm_call", "finalize")
research_builder.add_edge("finalize", END)

research_graph = research_builder.compile()

# TODO 6: 실행 및 검증
result = research_graph.invoke({"messages": [HumanMessage(content="LangGraph를 한 문장으로 설명해줘.")]})

print("result keys:", result.keys())
assert set(result.keys()) == {"result"}, f"output_schema가 올바르지 않습니다: {result.keys()}"
print("output_schema 검증 통과")
print("result:", result["result"])

result keys: dict_keys(['result'])
output_schema 검증 통과
result: LangGraph는 자연어 처리와 그래프 구조를 결합하여 언어 데이터를 효과적으로 분석하고 시각화하는 도구입니다.


---
## 문제 9. ReAct 에이전트 전체 구현

In [11]:
from tavily import TavilyClient
from langchain_core.messages import SystemMessage, ToolMessage

tavily_client = TavilyClient()

# TODO 1: simple_search tool 정의
@tool(parse_docstring=True)
def simple_search(query: str) -> str:
    """웹을 검색하고 결과를 반환한다.

    Args:
        query: 검색 쿼리
    """
    response = tavily_client.search(query=query, max_results=2)
    results = response["results"]
    return "\n\n".join([f"Title: {r['title']}\nContent: {r['content']}" for r in results])

# TODO 2: ReactState 정의
class ReactState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# TODO 3: 모델 + tool 바인딩
react_model = init_chat_model(model="openai:gpt-4o-mini")
react_model_with_tools = react_model.bind_tools([simple_search])

# TODO 4: llm_call 노드
def react_llm_call(state: ReactState):
    sys = SystemMessage(content="너는 웹 검색 에이전트야.")
    all_msgs = [sys] + list(state["messages"])
    response = react_model_with_tools.invoke(all_msgs)
    return {"messages": [response]}

# TODO 5: tool_node 구현
react_tools_by_name = {simple_search.name: simple_search}

def react_tool_node(state: ReactState):
    tool_calls = state["messages"][-1].tool_calls
    outputs = []
    for tc in tool_calls:
        result = react_tools_by_name[tc["name"]].invoke(tc["args"])
        outputs.append(ToolMessage(content=str(result), name=tc["name"], tool_call_id=tc["id"]))
    return {"messages": outputs}

# TODO 6: should_continue 라우팅 함수
def should_continue(state: ReactState) -> Literal["tool_node", "__end__"]:
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tool_node"
    return "__end__"

# TODO 7: 그래프 구성
react_builder = StateGraph(ReactState)
react_builder.add_node("llm_call", react_llm_call)
react_builder.add_node("tool_node", react_tool_node)
react_builder.add_edge(START, "llm_call")
react_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {"tool_node": "tool_node", "__end__": "__end__"}
)
react_builder.add_edge("tool_node", "llm_call")

react_graph = react_builder.compile()

# TODO 8: 실행
result = react_graph.invoke(
    {"messages": [HumanMessage(content="2026년 LangChain의 최신 버전을 검색해줘.")]}
)
print(result["messages"][-1].content)

MissingAPIKeyError: No API key provided. Please provide the api_key attribute or set the TAVILY_API_KEY environment variable.

---
## 문제 10. 멀티턴 대화 + compress 노드 구현

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import filter_messages

# TODO 1: MultiTurnState 정의
class MultiTurnState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    compressed_result: str

# TODO 2: research_node
def research_node(state: MultiTurnState):
    research_model = init_chat_model(model="openai:gpt-4o-mini")
    result = research_model.invoke(state["messages"])
    return {"messages": [result]}

# TODO 3: compress_node (filter_messages 사용)
compress_model = init_chat_model(model="google_genai:gemini-2.5-flash")

def compress_node(state: MultiTurnState):
    ai_only = filter_messages(state["messages"], include_types=["ai"])
    all_ai_text = "\n".join(msg.content for msg in ai_only)
    summary_prompt = f"다음 내용을 한 문장으로 요약해줘:\n{all_ai_text}"
    summary = compress_model.invoke([HumanMessage(content=summary_prompt)])
    return {"compressed_result": summary.content}

# TODO 4: 그래프 구성 + checkpointer 적용
checkpointer = InMemorySaver()

mt_builder = StateGraph(MultiTurnState)
mt_builder.add_node("research_node", research_node)
mt_builder.add_node("compress_node", compress_node)
mt_builder.add_edge(START, "research_node")
mt_builder.add_edge("research_node", "compress_node")
mt_builder.add_edge("compress_node", END)

mt_graph = mt_builder.compile(checkpointer=checkpointer)

# TODO 5: 멀티턴 실행
thread = {"configurable": {"thread_id": "turn-test"}}

result1 = mt_graph.invoke(
    {"messages": [HumanMessage(content="LangGraph의 주요 특징을 알려줘.")]},
    config=thread
)
print(f"Turn 1 messages: {len(result1['messages'])}개")

result2 = mt_graph.invoke(
    {"messages": [HumanMessage(content="그 중 가장 중요한 것은 뭐야?")]},
    config=thread
)
print(f"Turn 2 messages: {len(result2['messages'])}개")

# TODO 6: 검증
assert len(result2["messages"]) > len(result1["messages"]), \
    "멀티턴 대화에서 messages가 누적되지 않았습니다."
print("멀티턴 누적 확인")
print("Turn 2 compressed:", result2["compressed_result"])

ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-2.5-flash', 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error